<a href="https://colab.research.google.com/github/KuldeepIsharwal/Machine-Learning-Basics/blob/main/handling_missing_data_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Multivariate Imputation - imputation using more than one coln
two parts :


1.   KNN
2.   Iterative Imputer



# KNN - - Finds the k nearest rows and fills the missing value with the average of those neighbours' values.

-> uses nan - euclidian distance i.e, sqrt(weight*((x2-x1)^2+(y2-y1)^2))

where weight = total coordinates/no. of coordinates that are present

# Adv

 -more accurate

#DisAdv

 -High number of calculations

 -Model has to remember entire dataset to impute vals


In [109]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [110]:
df= pd.read_csv('/content/titanic.csv',usecols = ['Fare','Survived','Age'])

In [111]:
df.sample(5)

,Survived,Age,Fare
92,0,46.0,61.1750
507,1,NaN,26.5500
324,0,NaN,69.5500
154,0,NaN,7.3125
760,0,NaN,14.5000


In [112]:
df.isnull().mean()

,0
Survived,0.000000
Age,0.198653
Fare,0.000000


First train without knn and then with knn and comparing the accuracy score

In [113]:
x = df.drop(columns = ['Survived'])
y = df['Survived']

In [114]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 2)

In [115]:
knn = KNNImputer(n_neighbors=3,weights='distance')

x_train_trf = knn.fit_transform(x_train)
x_test_trf = knn.transform(x_test)
pd.DataFrame(x_train_trf, columns=x_train.columns)
pd.DataFrame(x_test_trf, columns=x_test.columns)

,Age,Fare
0,42.000000,26.2875
1,21.000000,8.0500
2,24.000000,65.0000
3,28.000000,56.4958
4,17.000000,7.9250
...,...,...
174,24.000000,8.0500
175,22.000000,9.0000
176,27.650794,69.5500
177,26.000000,7.8958


In [116]:
lr = LogisticRegression()

lr.fit(x_train_trf,y_train)
y_pred = lr.predict(x_test_trf)
accuracy_score(y_test,y_pred)

0.6145251396648045

In [117]:
#comparision with mean
si = SimpleImputer()

x_train_trf2 = si.fit_transform(x_train)
x_test_trf2 = si.transform(x_test)
pd.DataFrame(x_train_trf2, columns=x_train.columns)
pd.DataFrame(x_test_trf2, columns=x_test.columns)

,Age,Fare
0,42.000000,26.2875
1,21.000000,8.0500
2,24.000000,65.0000
3,28.000000,56.4958
4,17.000000,7.9250
...,...,...
174,24.000000,8.0500
175,22.000000,9.0000
176,29.785904,69.5500
177,26.000000,7.8958


In [118]:
lr = LogisticRegression()

lr.fit(x_train_trf2,y_train)
y_pred2 = lr.predict(x_test_trf2)
accuracy_score(y_test,y_pred2)

0.6145251396648045

#2 . MICE(Multiple Imputation by Chained Equations)
      -An iterative imputation algorithm that fills missing values by modelling each incomplete feature as a function of all other features — cycling through them repeatedly until the imputed values converge.

      -highly accurate but slow and memory heavy(dataset on server)


 Step 1 — Initial fill

Fill all missing values with a simple placeholder (usually mean/median) to get a complete dataset to start with.


Step 2 — Iterate column by column

For each column that had missing values:

Set its imputed values back to missing
Use all other columns as features in a regression model
Predict the missing values using that model
Fill them in



Step 3 — Repeat

Cycle through all incomplete columns again and again — each cycle uses the improved imputed values from the previous cycle.


Step 4 — Converge

After enough iterations (max_iter), imputed values stabilise and stop changing significantly.

In [120]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(
    max_iter=10,          # number of full cycles through all columns
    random_state=0,       # reproducibility
    initial_strategy='mean'  # how to fill values for the initial pass
)

imputer.fit(x_train)
x_train_imputed = imputer.transform(x_train)
x_test_imputed  = imputer.transform(x_test)    # reuse model learned from train

In [121]:
lr = LogisticRegression()

lr.fit(x_train_imputed,y_train)
y_pred2 = lr.predict(x_test_imputed)
accuracy_score(y_test,y_pred2)

0.6145251396648045